In [5]:
class RadixNode:
    def __init__(self, label="", is_end = False):
        self.children = {}
        self.label = label
        self.is_end = is_end
    
    def _get_common_prefix(self, word1, word2):
        low_bound = min(len(word1), len(word2))
        for idx in range(low_bound):
            if word1[idx] != word2[idx]:
                return idx
        return low_bound
    
    def _search(self, word):
        if not word:
            return self.is_end
        
        first_char = word[0]

        if first_char not in self.children:
            return False
        
        child = self.children[first_char]
        prefix_len = self._get_common_prefix(word, child.label)

        # if prefix_len equals word in tree then there's a full label match. we further recurse
        if prefix_len == len(child.label):
            return child._search(word[prefix_len:])
        
        # else it's probably a partial match or no match at all. We should terminate. A partial match if label was previously split
        # or word just doesn't match to the existing word in the tree.
        return False
    
    def _delete(self, word):
        if not word:
            if not self.is_end:
                return False
            self.is_end = False
            # Immediately delete current word if it doesn't have children else move to its child if it does in other to
            # keep track of other words that are in the path and then merge to the root node.
            return len(self.children) == 0
        
        first_char = word[0]
        if first_char not in self.children:
            return False
        
        child = self.children[first_char]
        prefix_len = self._get_common_prefix(word, child.label)

        # No match
        if prefix_len != len(child.label):
            return False
        
        is_prune_child = child._delete(word[prefix_len:])

        if is_prune_child:
            del self.children[first_char]
            
        elif first_char in self.children:
            """
            word = "er" tree = "toaster"
            e will exist as first_char in the children of "toast" which is "er".
            The idea is to get this child and merge to current node. 
            """
            self._apply_merge(self.children[first_char])
        
        return len(self.children) == 0 and not self.is_end
    
    def _apply_merge(self, node):
        """
        If current node isn't a leaf node and it has only one child, then merge this child's labels into the label of node
        """

        if not node.is_end and len(node.children) == 1:
            child_key = list(node.children.keys())[0]
            child = node.children[child_key]

            node.label += child.label
            node.children = child.children
            node.is_end = True

            # Child node object is garbage collected
    
    def _find_all_with_prefixes(self, prefix, visited_path, results):
        """ 
        This is broken into 2 parts
        1. Navigation phase: Recursively travesere the tree to find the entire path of the prefix
        2. Collection phase: Build the words that matches with the prefix after completing step 1.
        """

        # Navigation phase
        if prefix:
            first_char = prefix[0]

            if first_char not in self.children:
                return
            
            child = self.children[first_char]
            prefix_len = self._get_common_prefix(prefix, child.label)

            # The current edge completes the finding of the prefix
            if prefix_len == len(prefix):
                child._find_all_with_prefixes("", visited_path + child.label, results)
            elif prefix_len == len(child.label):
                child._find_all_with_prefixes(prefix[prefix_len:], visited_path + child.label, results)
            return
        
        # Collection phase
        if self.is_end:
            results.append(visited_path)
        
        for child in self.children.values():
            # prefix - first argument is set to "" because we are in the collection phase and only need to collect the labels
            child._find_all_with_prefixes("", visited_path + child.label, results)

class Radix:
    def __init__(self):
        # Initialize root node
        self.root = RadixNode(label="")
    
    def insert(self, word: RadixNode):
        current = self.root

        while word:
            first_char = word[0]

            if first_char not in current.children:
                node = RadixNode(word, True)
                current.children[first_char] = node
                return
            
            child = current.children[first_char]
            
            
            prefix_len = current._get_common_prefix(word, child.label)
            
            if prefix_len == len(child.label):
                word = word[prefix_len:]
                current = child
            else:
                # current = toad and word = toaster.
                split_node = RadixNode(label=child.label[:prefix_len])
                current.children[first_char] = split_node

                """current = toaster; word = toast
                set 'er' as label of old parent node
                point 'er' node to new parent node 'split_node'"""
                child.label = child.label[prefix_len:]
                split_node.children[child.label[0]] = child

                if prefix_len == len(word):
                    split_node.is_end = True
                else:
                    """current = toaster; word = toad
                    point remainder of word to new parent node
                    'd' becomes a node and its is_end is true"""
                    remaining_word = word[prefix_len:]
                    new_leaf_node = RadixNode(label=remaining_word, is_end=True)
                    split_node.children[remaining_word[0]] = new_leaf_node
                return
    
    def search(self, word):
        current = self.root
        return current._search(word)

    def delete(self, word):
        current = self.root
        return current._delete(word)
    
    def find_all_with_prefixes(self, prefix):
        results = []
        current = self.root
        current._find_all_with_prefixes(prefix, "", results)
        return results


In [6]:
import time


def run_tests():
    tree = Radix()
    
    print("--- Test 1: Basic Insert & Search ---")
    tree.insert("test")
    tree.insert("team")
    assert tree.search("test") == True, "Failed to find 'test'"
    assert tree.search("team") == True, "Failed to find 'team'"
    assert tree.search("tea") == False, "Found 'tea' which shouldn't exist yet"
    print("✅ Basic Insert Passed")

    print("\n--- Test 2: Split Logic (Case 3 & 4) ---")
    # Tree has "test", "team". Insert "toast" (New branch at root)
    tree.insert("toast")
    # Insert "toaster" (Extension of existing edge)
    tree.insert("toaster")
    # Insert "toad" (Fork split from "toast")
    tree.insert("toad")
    
    assert tree.search("toast") == True
    assert tree.search("toaster") == True
    assert tree.search("toad") == True
    assert tree.search("toas") == False
    print("✅ Split Logic Passed")

    print("\n--- Test 3: Prefix Search (Autocomplete) ---")
    # Should find: team, test, toast, toaster, toad
    results_t = sorted(tree.find_all_with_prefixes("t"))
    expected_t = ["team", "test", "toad", "toast", "toaster"]
    assert results_t == expected_t, f"Expected {expected_t}, got {results_t}"
    
    # Should find: toast, toaster, toad
    results_toa = sorted(tree.find_all_with_prefixes("toa"))
    expected_toa = ["toad", "toast", "toaster"]
    assert results_toa == expected_toa, f"Expected {expected_toa}, got {results_toa}"
    
    # "Tunneling" test: Search 'te' (should match inside 'team'/'test')
    results_te = sorted(tree.find_all_with_prefixes("te"))
    expected_te = ["team", "test"]
    assert results_te == expected_te, f"Expected {expected_te}, got {results_te}"
    print("✅ Prefix Search Passed")

    print("\n--- Test 4: Deletion & Merging ---")
    # Delete "toaster". "toast" should remain. 
    # Logic check: "toaster" was a child of "toast". 
    # After delete, "toast" has other children? No, "toaster" was extension of "toast".
    # Wait, "toad" and "toast" split at "toa". 
    # Structure: "toa" -> {"d" (toad), "st" (toast)}. 
    # "toast" -> "er" (toaster).
    
    tree.delete("toaster")
    assert tree.search("toaster") == False, "toaster should be gone"
    assert tree.search("toast") == True, "toast should still be there"
    
    # Delete "toast". Now "toa" node has child "d" (toad) and "st" (now empty leaf?).
    # Should merge "st" upwards or remove.
    tree.delete("toast")
    assert tree.search("toast") == False
    assert tree.search("toad") == True
    
    print("✅ Deletion Passed")
    
    print("\n--- Test 5: Edge Cases ---")
    tree.insert("a")
    tree.insert("ab")
    assert tree.search("a") == True
    assert tree.search("ab") == True
    tree.delete("a") # Delete prefix word
    assert tree.search("a") == False
    assert tree.search("ab") == True # Child should remain
    
    print("✅ Edge Cases Passed")
    print("\n🎉 ALL TESTS PASSED!")

# Start Timer
start_time = time.perf_counter()

ITERATIONS = 15
for _ in range(ITERATIONS):
    run_tests()

end_time = time.perf_counter()

total_time_ms = (end_time - start_time) * 1000
avg_time_ms = total_time_ms / ITERATIONS

print(f"Total time for {ITERATIONS} runs: {total_time_ms:.4f} ms")
print(f"Average time per run: {avg_time_ms:.6f} ms")



--- Test 1: Basic Insert & Search ---
✅ Basic Insert Passed

--- Test 2: Split Logic (Case 3 & 4) ---
✅ Split Logic Passed

--- Test 3: Prefix Search (Autocomplete) ---
✅ Prefix Search Passed

--- Test 4: Deletion & Merging ---
✅ Deletion Passed

--- Test 5: Edge Cases ---
✅ Edge Cases Passed

🎉 ALL TESTS PASSED!
--- Test 1: Basic Insert & Search ---
✅ Basic Insert Passed

--- Test 2: Split Logic (Case 3 & 4) ---
✅ Split Logic Passed

--- Test 3: Prefix Search (Autocomplete) ---
✅ Prefix Search Passed

--- Test 4: Deletion & Merging ---
✅ Deletion Passed

--- Test 5: Edge Cases ---
✅ Edge Cases Passed

🎉 ALL TESTS PASSED!
--- Test 1: Basic Insert & Search ---
✅ Basic Insert Passed

--- Test 2: Split Logic (Case 3 & 4) ---
✅ Split Logic Passed

--- Test 3: Prefix Search (Autocomplete) ---
✅ Prefix Search Passed

--- Test 4: Deletion & Merging ---
✅ Deletion Passed

--- Test 5: Edge Cases ---
✅ Edge Cases Passed

🎉 ALL TESTS PASSED!
--- Test 1: Basic Insert & Search ---
✅ Basic Insert Pa